In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import os
os.chdir('/content/drive/MyDrive/cs515-hw1b')

In [ ]:

!pip install -q ptflops

In [ ]:

!nvidia-smi

Sun Mar 22 10:23:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             27W /   70W |       0MiB /  15360MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!python main.py --experiment scratch --epochs 2 --lr 0.1 --device cuda

Device: cuda
100% 170M/170M [00:13<00:00, 12.9MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

[scratch] Model complexity:
MACs: 557.78 MMac | Params: 11.17 M

Epoch [1/2]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to

In [ ]:
!python main.py --experiment transfer_opt1 --epochs 30 --lr 0.01 --device cuda

Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 237MB/s]

[transfer_opt1] Model complexity:
MACs: 37.25 MMac | Params: 5.13 k

Epoch [1/30]
  [Batch 100/391] Loss: 2.2400  Acc@1: 24.22%
  [Batch 200/391] Loss: 2.1904  Acc@1: 20.31%
  [Batch 300/391] Loss: 2.2720  Acc@1: 20.31%
  Train Loss: 2.2877  Train Acc: 20.28%
  Test  Loss: 2.5653  Test  Acc: 10.71%
  Checkpoint saved → ./checkpoints/transfer_opt1_best.pth  (acc=10.71%)

Epoch [2/30]
  [Batch 100/391] Loss: 2.1716  Acc@1: 23.44%
  [Batch 200/391] Loss: 2.2264  Acc@1: 19.53%
  [Batch 300/391] Loss: 2.3041  Acc@1: 17.19%
  Train Loss: 2.2850  Train Acc: 20.81%
  Test  Loss: 2.5992  Test  Acc: 10.28%

Epoch [3/30]
  [Batch 100/391] Loss: 2.2524  Acc@1: 21.09%
  [Batch 200/391] Loss: 2.2284  Acc@1: 16.41%
  [Batch 300/391] Loss: 2.2602  Acc@1: 18.75%
  Train Loss: 2.2800  Train Acc: 20.90%
  Test  Loss: 2.4

In [ ]:
from parameters import DataConfig
from utils.dataset import get_dataloaders

cfg = DataConfig(resize=True, image_size=224, batch_size=4)
loader, _ = get_dataloaders(cfg)
imgs, labels = next(iter(loader))
print(imgs.shape)

torch.Size([4, 3, 32, 32])


In [ ]:
%%writefile utils/dataset.py
"""
utils/dataset.py
CIFAR-10 dataset loading and preprocessing utilities.
"""

from typing import Tuple
import torch
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

from parameters import DataConfig


def get_cifar10_transforms(resize: bool = False, image_size: int = 224) -> Tuple[transforms.Compose, transforms.Compose]:
    """
    Build train and test transforms for CIFAR-10.

    Args:
        resize: If True, resize images to image_size (for ImageNet-pretrained backbones).
        image_size: Target image size when resize=True.

    Returns:
        Tuple of (train_transform, test_transform).
    """
    normalize = transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010],
    )

    if resize:
        base_train = [
            transforms.Resize(image_size),
            transforms.RandomHorizontalFlip(),
        ]
        base_test = [transforms.Resize(image_size)]
    else:
        base_train = [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
        ]
        base_test = []

    train_transform = transforms.Compose(base_train + [transforms.ToTensor(), normalize])
    test_transform = transforms.Compose(base_test + [transforms.ToTensor(), normalize])

    return train_transform, test_transform


def get_dataloaders(cfg: DataConfig) -> Tuple[DataLoader, DataLoader]:
    """
    Create CIFAR-10 train and test DataLoaders.

    Args:
        cfg: DataConfig instance with dataset parameters.

    Returns:
        Tuple of (train_loader, test_loader).
    """
    train_transform, test_transform = get_cifar10_transforms(cfg.resize, cfg.image_size)

    train_set = torchvision.datasets.CIFAR10(
        root=cfg.data_dir, train=True, download=True, transform=train_transform
    )
    test_set = torchvision.datasets.CIFAR10(
        root=cfg.data_dir, train=False, download=True, transform=test_transform
    )

    train_loader = DataLoader(
        train_set, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=True
    )
    test_loader = DataLoader(
        test_set, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True
    )

    return train_loader, test_loader

Overwriting utils/dataset.py


In [ ]:
from importlib import reload
import utils.dataset
reload(utils.dataset)
from utils.dataset import get_dataloaders
from parameters import DataConfig

cfg = DataConfig(resize=True, image_size=224, batch_size=4)
loader, _ = get_dataloaders(cfg)
imgs, _ = next(iter(loader))
print(imgs.shape)  # (4, 3, 224, 224) çıkmalı

torch.Size([4, 3, 224, 224])


In [ ]:
!python main.py --experiment transfer_opt1 --epochs 30 --lr 0.01 --device cuda

Device: cuda

[transfer_opt1] Model complexity:
MACs: 37.25 MMac | Params: 5.13 k

Epoch [1/30]
  [Batch 100/391] Loss: 0.5988  Acc@1: 78.91%
  [Batch 200/391] Loss: 0.6153  Acc@1: 76.56%
  [Batch 300/391] Loss: 0.5406  Acc@1: 81.25%
  Train Loss: 0.7371  Train Acc: 74.86%
  Test  Loss: 0.5949  Test  Acc: 79.94%
  Checkpoint saved → ./checkpoints/transfer_opt1_best.pth  (acc=79.94%)

Epoch [2/30]
  [Batch 100/391] Loss: 0.5446  Acc@1: 80.47%
  [Batch 200/391] Loss: 0.5281  Acc@1: 78.91%
  [Batch 300/391] Loss: 0.7455  Acc@1: 70.31%
  Train Loss: 0.5962  Train Acc: 79.10%
  Test  Loss: 0.5846  Test  Acc: 80.33%
  Checkpoint saved → ./checkpoints/transfer_opt1_best.pth  (acc=80.33%)

Epoch [3/30]
  [Batch 100/391] Loss: 0.6007  Acc@1: 80.47%
  [Batch 200/391] Loss: 0.4626  Acc@1: 82.81%
  [Batch 300/391] Loss: 0.6542  Acc@1: 75.00%
  Train Loss: 0.5762  Train Acc: 79.94%
  Test  Loss: 0.5741  Test  Acc: 80.24%

Epoch [4/30]
  [Batch 100/391] Loss: 0.7039  Acc@1: 75.78%
  [Batch 200/391] 

In [ ]:
from importlib import reload
import utils.dataset
reload(utils.dataset)
from utils.dataset import get_dataloaders
from parameters import DataConfig

cfg = DataConfig(resize=False, batch_size=4)
loader, _ = get_dataloaders(cfg)
imgs, _ = next(iter(loader))
print(imgs.shape)  # (4, 3, 32, 32) çıkmalı

torch.Size([4, 3, 32, 32])


In [ ]:
!python main.py --experiment transfer_opt2 --epochs 30 --lr 0.01 --device cuda

Device: cuda

[transfer_opt2] Model complexity:
MACs: 557.78 MMac | Params: 11.17 M

Epoch [1/30]
  [Batch 100/391] Loss: 0.5707  Acc@1: 83.59%
  [Batch 200/391] Loss: 0.6593  Acc@1: 78.12%
  [Batch 300/391] Loss: 0.4665  Acc@1: 83.59%
  Train Loss: 0.7533  Train Acc: 73.91%
  Test  Loss: 0.3868  Test  Acc: 86.77%
  Checkpoint saved → ./checkpoints/transfer_opt2_best.pth  (acc=86.77%)

Epoch [2/30]
  [Batch 100/391] Loss: 0.3047  Acc@1: 88.28%
  [Batch 200/391] Loss: 0.2912  Acc@1: 88.28%
  [Batch 300/391] Loss: 0.3168  Acc@1: 90.62%
  Train Loss: 0.3354  Train Acc: 88.60%
  Test  Loss: 0.3107  Test  Acc: 89.66%
  Checkpoint saved → ./checkpoints/transfer_opt2_best.pth  (acc=89.66%)

Epoch [3/30]
  [Batch 100/391] Loss: 0.3452  Acc@1: 87.50%
  [Batch 200/391] Loss: 0.2055  Acc@1: 92.19%
  [Batch 300/391] Loss: 0.3640  Acc@1: 90.62%
  Train Loss: 0.2421  Train Acc: 91.67%
  Test  Loss: 0.2733  Test  Acc: 90.86%
  Checkpoint saved → ./checkpoints/transfer_opt2_best.pth  (acc=90.86%)

Epo

In [ ]:
!python main.py --experiment scratch --epochs 60 --lr 0.1 --device cuda

Device: cuda

[scratch] Model complexity:
MACs: 557.78 MMac | Params: 11.17 M

Epoch [1/60]
  [Batch 100/391] Loss: 1.8036  Acc@1: 30.47%
  [Batch 200/391] Loss: 1.6528  Acc@1: 38.28%
  [Batch 300/391] Loss: 1.8476  Acc@1: 36.72%
  Train Loss: 1.8520  Train Acc: 33.42%
  Test  Loss: 1.4337  Test  Acc: 46.30%
  Checkpoint saved → ./checkpoints/resnet_scratch_best.pth  (acc=46.30%)

Epoch [2/60]
  [Batch 100/391] Loss: 1.3404  Acc@1: 49.22%
  [Batch 200/391] Loss: 1.2979  Acc@1: 50.00%
  [Batch 300/391] Loss: 1.3584  Acc@1: 54.69%
  Train Loss: 1.3104  Train Acc: 52.16%
  Test  Loss: 1.2547  Test  Acc: 55.21%
  Checkpoint saved → ./checkpoints/resnet_scratch_best.pth  (acc=55.21%)

Epoch [3/60]
  [Batch 100/391] Loss: 0.9716  Acc@1: 63.28%
  [Batch 200/391] Loss: 1.0308  Acc@1: 64.84%
  [Batch 300/391] Loss: 1.1264  Acc@1: 61.72%
  Train Loss: 1.0213  Train Acc: 63.72%
  Test  Loss: 0.9143  Test  Acc: 68.53%
  Checkpoint saved → ./checkpoints/resnet_scratch_best.pth  (acc=68.53%)

Epoch 

In [ ]:
!python main.py --experiment scratch_ls --epochs 60 --lr 0.1 --device cuda

Device: cuda

[scratch_ls] Model complexity:
MACs: 557.78 MMac | Params: 11.17 M

Epoch [1/60]
  [Batch 100/391] Loss: 2.1113  Acc@1: 21.88%
  [Batch 200/391] Loss: 1.9749  Acc@1: 28.12%
  [Batch 300/391] Loss: 1.7234  Acc@1: 36.72%
  Train Loss: 2.1144  Train Acc: 28.82%
  Test  Loss: 1.5307  Test  Acc: 42.98%
  Checkpoint saved → ./checkpoints/resnet_scratch_ls_best.pth  (acc=42.98%)

Epoch [2/60]
  [Batch 100/391] Loss: 1.6220  Acc@1: 50.78%
  [Batch 200/391] Loss: 1.6219  Acc@1: 39.84%
  [Batch 300/391] Loss: 1.5622  Acc@1: 50.00%
  Train Loss: 1.5772  Train Acc: 49.69%
  Test  Loss: 1.3052  Test  Acc: 53.04%
  Checkpoint saved → ./checkpoints/resnet_scratch_ls_best.pth  (acc=53.04%)

Epoch [3/60]
  [Batch 100/391] Loss: 1.3044  Acc@1: 62.50%
  [Batch 200/391] Loss: 1.2422  Acc@1: 67.97%
  [Batch 300/391] Loss: 1.2461  Acc@1: 64.84%
  Train Loss: 1.3584  Train Acc: 61.16%
  Test  Loss: 1.0045  Test  Acc: 65.28%
  Checkpoint saved → ./checkpoints/resnet_scratch_ls_best.pth  (acc=65.

In [ ]:
!python main.py --experiment kd --epochs 30 --lr 0.01 --device cuda --teacher_ckpt checkpoints/resnet_scratch_ls_best.pth

Device: cpu

[kd] Model complexity:
MACs: 11.78 MMac | Params: 1.15 M
Teacher complexity:
MACs: 557.78 MMac | Params: 11.17 M

Epoch [1/30]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  [Batch 100/391] Loss: 1.5175  Acc@1: 32.03%
  [Batch 200/391] Loss: 1.4339  Acc@1: 33.59%
  [Batch 300/391] Loss: 1.3094  Acc@1: 45.31%
  Train Loss: 1.4106  Train Acc: 39.79%
  Test  Loss: 1.3690  Test  Acc: 55.08%
  Checkpoint saved → ./checkpoints/simplecnn_kd_best.pth  (acc=55.08%)

Epoch [2/30]
  [Batch 100/391] Loss: 1.2651  Acc@1: 39.84%
  [Batch 200/391] Loss: 1.2175  Acc@1: 50.78%
  [Batch 300/391] Loss: 1.1510  Acc@1: 53.91%
  Train Loss: 1.1959  Train Acc: 52.02%
  Test  Loss: 1.2259  Test  Acc: 60.89%
  Checkpoint saved → ./checkpoints/simplecnn_kd_best.pth  (acc=60.89%)

Epoch [3/30]
  [Batch 100/391] Loss: 1.0410 

In [ ]:
!python main.py --experiment soft_kd --epochs 30 --lr 0.01 --device cuda --teacher_ckpt checkpoints/resnet_scratch_ls_best.pth

Device: cuda

[soft_kd] Model complexity:
MACs: 26.06 MMac | Params: 2.24 M
Teacher complexity:
MACs: 557.78 MMac | Params: 11.17 M

Epoch [1/30]
  [Batch 100/391] Loss: 1.5882  Acc@1: 28.91%
  [Batch 200/391] Loss: 1.5201  Acc@1: 31.25%
  [Batch 300/391] Loss: 1.3901  Acc@1: 42.97%
  Train Loss: 1.5256  Train Acc: 31.68%
  Test  Loss: 1.6451  Test  Acc: 41.51%
  Checkpoint saved → ./checkpoints/mobilenet_softkd_best.pth  (acc=41.51%)

Epoch [2/30]
  [Batch 100/391] Loss: 1.3107  Acc@1: 45.31%
  [Batch 200/391] Loss: 1.1894  Acc@1: 54.69%
  [Batch 300/391] Loss: 1.2273  Acc@1: 50.78%
  Train Loss: 1.2981  Train Acc: 45.12%
  Test  Loss: 1.4346  Test  Acc: 50.59%
  Checkpoint saved → ./checkpoints/mobilenet_softkd_best.pth  (acc=50.59%)

Epoch [3/30]
  [Batch 100/391] Loss: 1.1990  Acc@1: 53.12%
  [Batch 200/391] Loss: 1.1497  Acc@1: 52.34%
  [Batch 300/391] Loss: 1.0506  Acc@1: 54.69%
  Train Loss: 1.1892  Train Acc: 50.64%
  Test  Loss: 1.3387  Test  Acc: 55.03%
  Checkpoint saved → .